# Regression Model

## Steps:

### Data Preparation
* Loading pre-stored sequences

### Model
* Check other optimizers
* Customize loss function

### Training
* Dropout
* Early Stopping
* Checkpoint
* Saving the models weights for further use
* Optuna

### Evaluation with Test Data
* Metrics per Horizon
* Plots per Horizon
* Overlay for specific time ranges



## Data preparation

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import sys, os

# Verify this prints True before importing
path = '/content/drive/MyDrive/RS - IEEE - ESA/notebooks/data_utils'

sys.path.insert(0, path)   # insert at 0, not append, to take priority

from data_preparation import train_val_test_split_by_date, scale, split_features_target, create_sequences, save_sequences

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import numpy as np
import os

In [4]:
# Loading the data
general_path = '/content/drive/My Drive/RS - IEEE - ESA/notebooks'  # ajusta la ruta
list_of_files = os.listdir(general_path)
list_of_parquet = [f for f in list_of_files if '.parquet' in f]
print(list_of_parquet)

dataframes = {}

for file in list_of_parquet:
  file_path = os.path.join(general_path, file)
  print(file)

  # if "precipitation" in file:
  #   continue

  df = pd.read_parquet(file_path)
  df_resampled = df.copy()

  print(df.info())

  if 'discharge' in str(file):
    df_resampled = df.resample('30min').mean()
  else:
    df_resampled = df.resample('30min').sum()

  file_prefix = file.replace('.parquet', '')
  df_resampled.columns = [f"{file_prefix}" for col in df_resampled.columns]

  # Store in dictionary
  dataframes[file] = df_resampled

# Merge using outer join to keep all timestamps
merged_df = pd.concat(dataframes.values(), axis=1, join='outer')

# Sort by index
merged_df = merged_df.sort_index()

# Dropping na
merged_df.dropna(inplace=True)

# Save dataset
dataset_name = "dataset.parquet"
general_path_parquet = '/content/drive/My Drive/RS - IEEE - ESA/notebooks'  # ajusta la ruta
os.makedirs(os.path.join(general_path_parquet, 'datasets'), exist_ok=True)
dataset_path = os.path.join(general_path_parquet, 'datasets', dataset_name)
merged_df.to_parquet(dataset_path)



['discharge_laa.parquet', 'discharge_klu.parquet', 'precipitation_bar.parquet', 'precipitation_sch.parquet']
discharge_laa.parquet
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 854313 entries, 2016-11-30 23:00:00+00:00 to 2025-12-31 22:55:00+00:00
Data columns (total 1 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   Abfluss (m³/s)  854313 non-null  float64
dtypes: float64(1)
memory usage: 13.0 MB
None
discharge_klu.parquet
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1375033 entries, 2009-12-31 23:00:00+00:00 to 2025-12-31 22:59:30+00:00
Data columns (total 1 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   Abfluss (m³/s)  1375033 non-null  float64
dtypes: float64(1)
memory usage: 21.0 MB
None
precipitation_bar.parquet
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 96007 entries, 2019-04-19 23:15:00+00:00 to 2022-01-20 11:00:00+00:00
Data columns (

In [5]:
# Loading the dataset
data = pd.read_parquet(dataset_path)
data.dropna()

# Splitting data into train, val, test
train, val, test = train_val_test_split_by_date(
  data,
  train_end_date='2021-01-01',
  val_end_date='2021-07-01')

# Scaling the data
train_scaled, val_scaled, test_scaled, scaler_train = scale(train, val, test, method='standard')

# Splitting into X,y
target_column='discharge_klu'
X_train, y_train = split_features_target(train_scaled, target_column)
X_val, y_val = split_features_target(val_scaled, target_column)
X_test, y_test = split_features_target(test_scaled, target_column)

# Creating sequences
lookback = 12 # 12 Timestamps --> Each timestamp 15 min --> 3 hours into the past
horizon = 8 # 8 Timestamps --> 2 hours into the future
X_train_seq, y_train_seq = create_sequences(X_train, y_train, lookback, horizon)
X_val_seq, y_val_seq = create_sequences(X_val, y_val, lookback, horizon)
X_test_seq, y_test_seq = create_sequences(X_test, y_test, lookback, horizon)

# Saving sequences
save_dir = os.path.join(general_path_parquet, 'sequences')
os.makedirs(save_dir, exist_ok=True)
save_sequences(X_train_seq, y_train_seq, X_val_seq, y_val_seq, X_test_seq, y_test_seq, scaler_train, save_dir)

In [6]:
print("First sequence of X_train:")
print(X_train_seq[0])
print(f"\nShape: {X_train_seq[0].shape}")

First sequence of X_train:
[[-0.41355397 -0.13073671 -0.12395607]
 [-0.40589529 -0.13073671 -0.12395607]
 [-0.41738331 -0.13073671 -0.12395607]
 [-0.41451131 -0.13073671 -0.12395607]
 [-0.41323486 -0.13073671 -0.12395607]
 [-0.47201525 -0.13073671 -0.12395607]
 [-0.4116393  -0.13073671 -0.12395607]
 [-0.41004374 -0.13073671 -0.12395607]
 [-0.40780996 -0.13073671 -0.12395607]
 [-0.53194444 -0.13073671 -0.12395607]
 [-0.52843421 -0.13073671 -0.12395607]
 [-0.55141026 -0.13073671 -0.12395607]]

Shape: (12, 3)


In [7]:
print("First sequence of X_train:")
print(X_train_seq[0])
print(f"\nShape: {X_train_seq[0].shape}")

First sequence of X_train:
[[-0.41355397 -0.13073671 -0.12395607]
 [-0.40589529 -0.13073671 -0.12395607]
 [-0.41738331 -0.13073671 -0.12395607]
 [-0.41451131 -0.13073671 -0.12395607]
 [-0.41323486 -0.13073671 -0.12395607]
 [-0.47201525 -0.13073671 -0.12395607]
 [-0.4116393  -0.13073671 -0.12395607]
 [-0.41004374 -0.13073671 -0.12395607]
 [-0.40780996 -0.13073671 -0.12395607]
 [-0.53194444 -0.13073671 -0.12395607]
 [-0.52843421 -0.13073671 -0.12395607]
 [-0.55141026 -0.13073671 -0.12395607]]

Shape: (12, 3)
